In [3]:
import sqlite3
import statistics
from transformers import AutoTokenizer

DB_PATH = 'WAMASRAGAGENT.db'
TABLE_NAME = 'document_chunks'
COL_TEXT = 'text_content'
COL_FILENAME = 'filename'
COL_CHUNK_ID = 'chunk_index'

TOKENIZER_MODEL = "Qwen/Qwen2.5-32B-Instruct"

def analyze_tokens():

    try:
        conn = sqlite3.connect(DB_PATH)
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()

        query = f"""
        SELECT {COL_FILENAME}, {COL_CHUNK_ID}, {COL_TEXT}
        FROM {TABLE_NAME}
        """
        cursor.execute(query)
        rows = cursor.fetchall()

    except sqlite3.Error as e:
        print(f"Errore Database: {e}")
        return
    finally:
        if conn:
            conn.close()

    if not rows:
        print("Nessun dato trovato nella tabella.")
        return

    # Caricamento tokenizer Qwen
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            TOKENIZER_MODEL,
            trust_remote_code=True,
            use_fast=True
        )
    except Exception as e:
        print(f"Errore nel caricamento del tokenizer Qwen: {e}")
        return

    chunk_stats = []

    print(f"Analisi di {len(rows)} righe in corso...")

    for row in rows:
        text = row[COL_TEXT]

        if not text:
            token_count = 0
        else:
            token_count = len(tokenizer.encode(text, add_special_tokens=False))

        chunk_stats.append({
            'file': row[COL_FILENAME],
            'id': row[COL_CHUNK_ID],
            'tokens': token_count,
            'preview': text[:50].replace('\n', ' ') + "..." if text else ""
        })

    if not chunk_stats:
        return

    tokens_list = [c['tokens'] for c in chunk_stats]

    avg_tokens = statistics.mean(tokens_list)
    max_chunk = max(chunk_stats, key=lambda x: x['tokens'])
    min_chunk = min(chunk_stats, key=lambda x: x['tokens'])

    print("-" * 40)
    print("📊 REPORT STATISTICHE TOKEN (Qwen 2.5)")
    print("-" * 40)
    print(f"Totale chunk analizzati: {len(chunk_stats)}")
    print(f"Media token per chunk:   {avg_tokens:.2f}")
    print("-" * 40)

    print("🔼 CHUNK PIÙ GRANDE:")
    print(f"   File:   {max_chunk['file']}")
    print(f"   ID:     {max_chunk['id']}")
    print(f"   Token:  {max_chunk['tokens']}")
    print(f"   Intro:  {max_chunk['preview']}")
    print("-" * 40)

    print("🔽 CHUNK PIÙ PICCOLO:")
    print(f"   File:   {min_chunk['file']}")
    print(f"   ID:     {min_chunk['id']}")
    print(f"   Token:  {min_chunk['tokens']}")
    print(f"   Intro:  {min_chunk['preview']}")
    print("-" * 40)

if __name__ == "__main__":
    analyze_tokens()


Analisi di 140 righe in corso...
----------------------------------------
📊 REPORT STATISTICHE TOKEN (Qwen 2.5)
----------------------------------------
Totale chunk analizzati: 140
Media token per chunk:   190.26
----------------------------------------
🔼 CHUNK PIÙ GRANDE:
   File:   PUB_Creazione_nuove_UDC.pdf
   ID:     7
   Token:  244
   Intro:  desiderato - -Quantità : selezionare la quantità d...
----------------------------------------
🔽 CHUNK PIÙ PICCOLO:
   File:   UTL_Assegnazioni_ordini_utente.pdf
   ID:     3
   Token:  34
   Intro:  La tabella include colonne con dettagli come data,...
----------------------------------------


In [22]:
from google import genai
import os
from docling.datamodel.document import DoclingDocument
from dotenv import load_dotenv
load_dotenv()

# Configurazione Gemini API
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

nomefile = "UTL_Refresh_Reset_OT"

DOC_SOURCE = f"preprocessing/scratch/{nomefile}/{nomefile}.json"

# Testo di esempio
full_text = DoclingDocument.load_from_json(DOC_SOURCE).export_to_markdown()

# Conta i token con Gemini 2.5 Flash
token_count = client.models.count_tokens(model="gemini-2.5-flash", contents=full_text)
print(f"Total tokens in {nomefile}: {token_count.total_tokens}")

Total tokens in UTL_Refresh_Reset_OT: 2040
